# 4 — GRU Comparison Across Cleaning Methods

Trains a GRU (fixed hyperparameters, 5 runs each) on five training sets to see
which of notebook 3's cleaning methods improves detection skill over no
cleaning at all.

| dataset | what it is |
|---|---|
| `baseline` | hybrid-normalized, best-channel data (notebooks 1 and 2), no cleaning |
| `deep_enn` | + Deep ENN |
| `deep_tomek` | + Deep Tomek Links |
| `deep_ae` | + Autoencoder novelty detection |
| `siamese` | + Siamese/Triplet network |

Val and test are identical across all five, since cleaning only touches the
training set, so this compares training-set choice alone.

**Input:** `./final_split_data_HybridNorm_BestFeatures{,_DeepENN,_DeepTomek,_DeepAE,_Siamese}`

**Output:** `./results/gru_comparison.txt` — raw and calibrated metrics per run
per dataset — plus a printed ranking.

Hyperparameters, calibration, and metrics match notebooks 1 to 3
(`units=64, layers=1, dropout=0.1, lr=0.001, batch_size=64`, Platt scaling with
`class_weight='balanced'` fitted on the validation split), so results are
directly comparable.

**Run after:** notebook 3.


## 1. Load All Five Datasets

In [1]:
import pickle
import numpy as np
import os

def load_split(path):
    with open(path, "rb") as f:
        data = pickle.load(f)
    return data["X"].astype(np.float32), data["y"]

def load_dataset(folder):
    X_train, y_train = load_split(f"{folder}/train_set.pkl")
    X_val,   y_val   = load_split(f"{folder}/val_set.pkl")
    X_test,  y_test  = load_split(f"{folder}/test_set.pkl")
    return {
        "X_train": X_train, "y_train": y_train,
        "X_val":   X_val,   "y_val":   y_val,
        "X_test":  X_test,  "y_test":  y_test,
    }

dataset_folders = {
    "baseline":   "./final_split_data_HybridNorm_BestFeatures",
    "deep_enn":   "./final_split_data_HybridNorm_BestFeatures_DeepENN",
    "deep_tomek": "./final_split_data_HybridNorm_BestFeatures_DeepTomek",
    "deep_ae":    "./final_split_data_HybridNorm_BestFeatures_DeepAE",
    "siamese":    "./final_split_data_HybridNorm_BestFeatures_Siamese",
}

datasets = {name: load_dataset(folder) for name, folder in dataset_folders.items()}

print(f"{'Dataset':<14} {'Train shape':<18} {'Non-SEP':>9} {'SEP':>6} {'Val shape':<16} {'Test shape':<16}")
print("─" * 90)
for name, d in datasets.items():
    counts = np.bincount(d["y_train"].astype(int), minlength=2)
    print(
        f"{name:<14} {str(d['X_train'].shape):<18} {counts[0]:>9} {counts[1]:>6} "
        f"{str(d['X_val'].shape):<16} {str(d['X_test'].shape):<16}"
    )

Dataset        Train shape          Non-SEP    SEP Val shape        Test shape      
──────────────────────────────────────────────────────────────────────────────────────────
baseline       (12455, 288, 3)        12337    118 (1780, 288, 3)   (3559, 288, 3)  
deep_enn       (12383, 288, 3)        12265    118 (1780, 288, 3)   (3559, 288, 3)  
deep_tomek     (12355, 288, 3)        12237    118 (1780, 288, 3)   (3559, 288, 3)  
deep_ae        (12061, 288, 3)        11943    118 (1780, 288, 3)   (3559, 288, 3)  
siamese        (12387, 288, 3)        12269    118 (1780, 288, 3)   (3559, 288, 3)  


## 2. GRU Helper Functions

In [2]:
import torch
import torch.nn as nn
import time
from sklearn.metrics import confusion_matrix, brier_score_loss
import warnings
warnings.filterwarnings('ignore')

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")


def compute_metrics(y_true, y_pred):
    cm             = confusion_matrix(y_true, y_pred)
    TN, FP, FN, TP = cm.ravel()

    # Probability of Detection (PoD), MEMPSEP Section 9.2 item 1.
    # Identical formula to recall -- kept as a separate named field so
    # printed output matches the paper's terminology.
    pod       = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    recall    = pod
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    accuracy  = (TP + TN) / (TP + TN + FP + FN)

    tss  = recall - FP / (FP + TN) if (FP + TN) > 0 else 0.0

    # HSS1 (Barnes & Leka 2008): (TP + TN - N) / P  ==  (TP - FP) / (TP + FN)
    hss1 = (TP - FP) / (TP + FN) if (TP + FN) > 0 else 0.0

    denom = ((TP + FN) * (FN + TN) + (TP + FP) * (FP + TN))
    hss2  = 2 * (TP * TN - FP * FN) / denom if denom > 0 else 0.0

    hits_random = (TP + FP) * (TP + FN) / (TP + TN + FP + FN)
    gss = (TP - hits_random) / (TP + FP + FN - hits_random) if (TP + FP + FN - hits_random) > 0 else 0.0

    far = FP / (TP + FP) if (TP + FP) > 0 else float('nan')
    bias = (TP + FP) / (TP + FN) if (TP + FN) > 0 else 0.0

    return {
        'TP': int(TP), 'TN': int(TN), 'FP': int(FP), 'FN': int(FN),
        'tss': tss, 'far': far, 'bias': bias,
        'hss1': hss1, 'hss2': hss2, 'gss': gss,
        'pod': pod, 'recall': recall, 'precision': precision, 'f1': f1, 'accuracy': accuracy
    }


class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1, dropout=0.3):
        super(GRUModel, self).__init__()
        self.gru      = nn.GRU(input_size, hidden_size, num_layers=num_layers,
                               batch_first=True,
                               dropout=dropout if num_layers > 1 else 0)
        self.bn       = nn.BatchNorm1d(hidden_size)
        self.dropout  = nn.Dropout(dropout)
        self.fc1      = nn.Linear(hidden_size, 32)
        self.relu     = nn.ReLU()
        self.dropout2 = nn.Dropout(0.2)
        self.fc2      = nn.Linear(32, 1)

    def forward(self, x):
        out, _ = self.gru(x)
        out    = out[:, -1, :]
        out    = self.bn(out)
        out    = self.dropout(out)
        out    = self.relu(self.fc1(out))
        out    = self.dropout2(out)
        return self.fc2(out).squeeze()


def train_and_evaluate_gru(params, X_train, y_train, X_val, y_val, X_test, y_test, class_ratio=13):
    X_tr = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_tr = torch.tensor(y_train, dtype=torch.float32).to(device)
    X_va = torch.tensor(X_val,   dtype=torch.float32).to(device)
    y_va = torch.tensor(y_val,   dtype=torch.float32).to(device)
    X_te = torch.tensor(X_test,  dtype=torch.float32).to(device)

    input_size = X_train.shape[2]
    model      = GRUModel(input_size,
                          hidden_size = params["units"],
                          num_layers  = params["layers"],
                          dropout     = params["dropout"]).to(device)

    pos_weight = torch.tensor([float(class_ratio)], dtype=torch.float32).to(device)
    criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer  = torch.optim.Adam(model.parameters(), lr=params["lr"])

    batch_size     = params["batch_size"]
    n_samples      = X_tr.shape[0]
    best_val_loss  = float('inf')
    patience_count = 0
    best_state     = None

    t0 = time.time()
    for epoch in range(50):
        model.train()
        perm = torch.randperm(n_samples)
        for i in range(0, n_samples, batch_size):
            idx    = perm[i:i + batch_size]
            xb, yb = X_tr[idx], y_tr[idx]
            optimizer.zero_grad()
            loss   = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(X_va), y_va).item()

        if val_loss < best_val_loss:
            best_val_loss  = val_loss
            patience_count = 0
            best_state     = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            patience_count += 1
            if patience_count >= 5:
                break

    train_time = time.time() - t0

    model.load_state_dict(best_state)
    model.eval()

    # Raw (uncalibrated) sigmoid outputs -- needed both for evaluation and,
    # for the validation split, to fit a probability calibrator downstream.
    with torch.no_grad():
        y_val_prob = torch.sigmoid(model(X_va)).cpu().numpy()

    t0 = time.time()
    with torch.no_grad():
        y_prob = torch.sigmoid(model(X_te)).cpu().numpy()
    y_pred     = (y_prob >= 0.5).astype(int)
    infer_time = time.time() - t0

    metrics = compute_metrics(y_test.astype(int), y_pred)
    metrics['brier']      = brier_score_loss(y_test, y_prob)
    metrics['train_time'] = train_time
    metrics['infer_time'] = infer_time

    del model, optimizer, criterion, X_tr, y_tr, X_va, y_va, X_te, pos_weight
    if device.type == "mps":
        torch.mps.empty_cache()

    return metrics, y_val_prob, y_prob

Using device: mps


## 3. Train & Evaluate GRU on Each Dataset

Same protocol as notebooks 1–3: fixed best hyperparameters, 5 runs per
dataset, Platt-scaling calibration (`class_weight='balanced'`, fit on the
validation set) after each run, raw + calibrated metrics printed per run
and averaged per dataset at the end.

In [3]:
import copy
from sklearn.linear_model import LogisticRegression

best_params = {
    "units": 64,
    "layers": 1,
    "dropout": 0.1,
    "lr": 0.001,
    "batch_size": 64
}

N_RUNS = 5

avg_keys = ['tss', 'far', 'bias', 'hss1', 'hss2', 'gss',
            'pod', 'recall', 'precision', 'f1', 'accuracy', 'brier']

def avg_metrics(metrics_list, keys):
    return {k: np.nanmean([m[k] for m in metrics_list]) for k in keys}

all_results = {}

for dataset_name, d in datasets.items():

    X_train_d, y_train_d = d["X_train"], d["y_train"]
    X_val_d,   y_val_d   = d["X_val"],   d["y_val"]
    X_test_d,  y_test_d  = d["X_test"],  d["y_test"]

    train_counts_d = np.bincount(y_train_d.astype(int), minlength=2)
    class_ratio_d  = train_counts_d[0] / train_counts_d[1]

    print("═" * 70)
    print(f"  Dataset    : {dataset_name}")
    print(f"  Params     : {best_params}")
    print(f"  Train      : {X_train_d.shape} | Non-SEP={train_counts_d[0]} | SEP={train_counts_d[1]}")
    print(f"  Ratio      : {class_ratio_d:.1f}:1")
    print(f"  Runs       : {N_RUNS}")
    print("═" * 70)

    raw_metrics_list = []
    cal_metrics_list = []

    for run in range(1, N_RUNS + 1):

        print(f"\n{'─' * 70}")
        print(f"[{dataset_name}] Run {run}/{N_RUNS}")
        print(f"{'─' * 70}")

        metrics, y_val_prob, y_test_prob = train_and_evaluate_gru(
            copy.deepcopy(best_params),
            X_train_d, y_train_d,
            X_val_d,   y_val_d,
            X_test_d,  y_test_d,
            class_ratio=class_ratio_d
        )

        # Platt scaling, fit on the validation set, class-balanced (see
        # notebook 1 for why an unweighted calibrator collapses recall to 0
        # under this level of class imbalance).
        calibrator = LogisticRegression(class_weight='balanced')
        calibrator.fit(y_val_prob.reshape(-1, 1), y_val_d.astype(int))

        y_test_prob_cal = calibrator.predict_proba(y_test_prob.reshape(-1, 1))[:, 1]
        y_test_pred_cal = (y_test_prob_cal >= 0.5).astype(int)

        metrics_cal = compute_metrics(y_test_d.astype(int), y_test_pred_cal)
        metrics_cal['brier'] = brier_score_loss(y_test_d, y_test_prob_cal)

        raw_metrics_list.append(metrics)
        cal_metrics_list.append(metrics_cal)

        print("Raw (uncalibrated):")
        print(f"  TP={metrics['TP']}  TN={metrics['TN']}  FP={metrics['FP']}  FN={metrics['FN']}")
        print(f"  TSS={metrics['tss']:.4f}  FAR={metrics['far']:.4f}  Bias={metrics['bias']:.4f}")
        print(f"  HSS1={metrics['hss1']:.4f}  HSS2={metrics['hss2']:.4f}  GSS={metrics['gss']:.4f}")
        print(f"  PoD={metrics['pod']:.4f}  Recall={metrics['recall']:.4f}  Precision={metrics['precision']:.4f}  F1={metrics['f1']:.4f}  Acc={metrics['accuracy']:.4f}")
        print(f"  Brier={metrics['brier']:.4f}  Train={metrics['train_time']:.2f}s  Infer={metrics['infer_time']:.4f}s")

        print("Calibrated (Platt scaling, fit on val set):")
        print(f"  TP={metrics_cal['TP']}  TN={metrics_cal['TN']}  FP={metrics_cal['FP']}  FN={metrics_cal['FN']}")
        print(f"  TSS={metrics_cal['tss']:.4f}  FAR={metrics_cal['far']:.4f}  Bias={metrics_cal['bias']:.4f}")
        print(f"  HSS1={metrics_cal['hss1']:.4f}  HSS2={metrics_cal['hss2']:.4f}  GSS={metrics_cal['gss']:.4f}")
        print(f"  PoD={metrics_cal['pod']:.4f}  Recall={metrics_cal['recall']:.4f}  Precision={metrics_cal['precision']:.4f}  F1={metrics_cal['f1']:.4f}  Acc={metrics_cal['accuracy']:.4f}")
        print(f"  Brier={metrics_cal['brier']:.4f}")

    raw_avg = avg_metrics(raw_metrics_list, avg_keys)
    cal_avg = avg_metrics(cal_metrics_list, avg_keys)

    all_results[dataset_name] = {
        "raw_list": raw_metrics_list,
        "cal_list": cal_metrics_list,
        "raw_avg": raw_avg,
        "cal_avg": cal_avg,
        "train_shape": X_train_d.shape,
    }

    print(f"\n[{dataset_name}] Average of {N_RUNS} runs:")
    print("  Raw       :", {k: round(v, 4) for k, v in raw_avg.items()})
    print("  Calibrated:", {k: round(v, 4) for k, v in cal_avg.items()})

print("\nAll datasets evaluated.")

══════════════════════════════════════════════════════════════════════
  Dataset    : baseline
  Params     : {'units': 64, 'layers': 1, 'dropout': 0.1, 'lr': 0.001, 'batch_size': 64}
  Train      : (12455, 288, 3) | Non-SEP=12337 | SEP=118
  Ratio      : 104.6:1
  Runs       : 5
══════════════════════════════════════════════════════════════════════

──────────────────────────────────────────────────────────────────────
[baseline] Run 1/5
──────────────────────────────────────────────────────────────────────
Raw (uncalibrated):
  TP=27  TN=3081  FP=444  FN=7
  TSS=0.6682  FAR=0.9427  Bias=13.8529
  HSS1=-12.2647  HSS2=0.0907  GSS=0.0475
  PoD=0.7941  Recall=0.7941  Precision=0.0573  F1=0.1069  Acc=0.8733
  Brier=0.0954  Train=158.12s  Infer=0.3263s
Calibrated (Platt scaling, fit on val set):
  TP=27  TN=3027  FP=498  FN=7
  TSS=0.6528  FAR=0.9486  Bias=15.4412
  HSS1=-13.8529  HSS2=0.0801  GSS=0.0417
  PoD=0.7941  Recall=0.7941  Precision=0.0514  F1=0.0966  Acc=0.8581
  Brier=0.1102

─

## 4. Comparison Summary

In [4]:
print("═" * 110)
print("  Comparison -- GRU (5 runs each) across cleaning methods, ranked by calibrated TSS")
print("═" * 110)
print(
    f"{'Dataset':<14} {'Train N':>9} {'Raw TSS':>9} {'Cal TSS':>9} "
    f"{'Raw PoD':>9} {'Cal PoD':>9} {'Raw Brier':>10} {'Cal Brier':>10}"
)
print("─" * 110)

ranked = sorted(all_results.items(), key=lambda kv: kv[1]["cal_avg"]["tss"], reverse=True)

for name, r in ranked:
    print(
        f"{name:<14} {r['train_shape'][0]:>9} "
        f"{r['raw_avg']['tss']:>9.4f} {r['cal_avg']['tss']:>9.4f} "
        f"{r['raw_avg']['pod']:>9.4f} {r['cal_avg']['pod']:>9.4f} "
        f"{r['raw_avg']['brier']:>10.4f} {r['cal_avg']['brier']:>10.4f}"
    )

best_name, best_r = ranked[0]
print(f"\nBest cleaning method (by avg calibrated TSS): {best_name}")
print(f"  Calibrated TSS   : {best_r['cal_avg']['tss']:.4f}")
print(f"  Calibrated PoD   : {best_r['cal_avg']['pod']:.4f}")
print(f"  Calibrated Brier : {best_r['cal_avg']['brier']:.4f}")

if "baseline" in all_results and best_name != "baseline":
    baseline_cal_tss = all_results["baseline"]["cal_avg"]["tss"]
    delta = best_r["cal_avg"]["tss"] - baseline_cal_tss
    print(f"\nImprovement over baseline (no extra cleaning): {delta:+.4f} TSS "
          f"({baseline_cal_tss:.4f} -> {best_r['cal_avg']['tss']:.4f})")
elif best_name == "baseline":
    print("\nNone of the four cleaning methods beat the baseline (no extra "
          "cleaning) on average calibrated TSS.")

══════════════════════════════════════════════════════════════════════════════════════════════════════════════
  Comparison -- GRU (5 runs each) across cleaning methods, ranked by calibrated TSS
══════════════════════════════════════════════════════════════════════════════════════════════════════════════
Dataset          Train N   Raw TSS   Cal TSS   Raw PoD   Cal PoD  Raw Brier  Cal Brier
──────────────────────────────────────────────────────────────────────────────────────────────────────────────
siamese            12387    0.6825    0.6778    0.8059    0.8118     0.1025     0.1116
deep_tomek         12355    0.6724    0.6642    0.8059    0.8059     0.1086     0.1155
deep_ae            12061    0.6786    0.6636    0.8059    0.8118     0.1005     0.1182
deep_enn           12383    0.6637    0.6632    0.8000    0.8059     0.1165     0.1182
baseline           12455    0.6539    0.6480    0.7824    0.7941     0.1067     0.1187

Best cleaning method (by avg calibrated TSS): siamese
  Cali

## 5. Save Results

In [5]:
import os

RESULTS_DIR = "./results"
os.makedirs(RESULTS_DIR, exist_ok=True)


def save_metrics_csv(filepath, rows, extra_cols=()):
    header = (",".join(extra_cols) + ("," if extra_cols else "") +
              "run,kind,TP,TN,FP,FN,tss,hss1,hss2,gss,pod,recall,precision,"
              "f1,accuracy,brier,far,bias,train_time,infer_time")
    with open(filepath, "w") as f:
        f.write(header + "\n")
        for r in rows:
            prefix = (",".join(str(r[c]) for c in extra_cols) + ",") if extra_cols else ""
            f.write(
                prefix +
                f"{r['run']},{r['kind']},"
                f"{r['TP']},{r['TN']},{r['FP']},{r['FN']},"
                f"{r['tss']:.6f},{r['hss1']:.6f},{r['hss2']:.6f},{r['gss']:.6f},"
                f"{r['pod']:.6f},{r['recall']:.6f},{r['precision']:.6f},{r['f1']:.6f},{r['accuracy']:.6f},"
                f"{r['brier']:.6f},{r['far']:.6f},{r['bias']:.6f},"
                f"{r.get('train_time', float('nan')):.4f},{r.get('infer_time', float('nan')):.4f}\n"
            )
    print(f"Saved: {filepath}")


rows = []
for dataset_name, r in all_results.items():
    for i, m in enumerate(r["raw_list"], start=1):
        rows.append({**m, "dataset": dataset_name, "run": i, "kind": "raw"})
    for i, m in enumerate(r["cal_list"], start=1):
        rows.append({**m, "dataset": dataset_name, "run": i, "kind": "calibrated"})

save_metrics_csv(os.path.join(RESULTS_DIR, "gru_comparison.txt"), rows, extra_cols=("dataset",))

Saved: ./results/gru_comparison.txt
